In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

print('All imports successful.')

All imports successful.


In [46]:
# File path
ckd_path = "../data/chronic_kidney_disease/kidney_disease.csv"

# Load dataset
ckd_df = pd.read_csv(ckd_path)

# Show basic information
print("CKD dataset loaded successfully")
print("Shape of dataset:", ckd_df.shape)

# Display first 5 rows
ckd_df.head()

CKD dataset loaded successfully
Shape of dataset: (400, 26)


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [47]:
# Step 4: Check structure before cleaning
print("Column names:")
print(ckd_df.columns.tolist())

print("\nData types before cleaning:")
print(ckd_df.dtypes)

print("\nMissing values before cleaning:")
print(ckd_df.isnull().sum())

Column names:
['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']

Data types before cleaning:
id                  int64
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc                   str
pc                    str
pcc                   str
ba                    str
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv                   str
wc                    str
rc                    str
htn                   str
dm                    str
cad                   str
appet                 str
pe                    str
ane                   str
classification        str
dtype: object

Missing values before cleaning:
id                  0
age                 9
bp    

In [48]:
# Replace dirty missing values like ? with NaN
ckd_df = ckd_df.replace(r'^\s*\?\s*$', np.nan, regex=True)

print("Replaced '?' values with NaN")

Replaced '?' values with NaN


In [49]:
# Remove leading/trailing spaces from text columns
for col in ckd_df.select_dtypes(include=['object']).columns:
    ckd_df[col] = ckd_df[col].astype(str).str.strip()
    ckd_df[col] = ckd_df[col].replace("nan", np.nan)

print("Whitespace cleaned from object columns")

Whitespace cleaned from object columns


In [50]:
# Drop ID column because it is not useful for prediction
if 'id' in ckd_df.columns:
    ckd_df.drop(columns=['id'], inplace=True)

print("Updated shape after dropping id:", ckd_df.shape)

Updated shape after dropping id: (400, 25)


In [51]:
# Convert numeric-like columns stored as object into numeric type
numeric_like_cols = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']

for col in numeric_like_cols:
    if col in ckd_df.columns:
        ckd_df[col] = pd.to_numeric(ckd_df[col], errors='coerce')

print("Numeric-like columns converted")

Numeric-like columns converted


In [52]:
# Standardize text values to lowercase and clean spaces
for col in ckd_df.select_dtypes(include=['object']).columns:
    ckd_df[col] = ckd_df[col].str.lower().str.strip()

# Fix known inconsistent values
if 'dm' in ckd_df.columns:
    ckd_df['dm'] = ckd_df['dm'].replace({
        '\tyes': 'yes',
        '\tno': 'no',
        ' yes': 'yes',
        ' no': 'no'
    })

if 'cad' in ckd_df.columns:
    ckd_df['cad'] = ckd_df['cad'].replace({
        '\tyes': 'yes',
        '\tno': 'no'
    })

if 'classification' in ckd_df.columns:
    ckd_df['classification'] = ckd_df['classification'].replace({
        'ckd\t': 'ckd',
        'not ckd': 'notckd'
    })

print("Text values standardized")

Text values standardized


In [53]:
# Encode categorical columns using explicit mapping
mapping_dict = {
    'rbc': {'normal': 0, 'abnormal': 1},
    'pc': {'normal': 0, 'abnormal': 1},
    'pcc': {'notpresent': 0, 'present': 1},
    'ba': {'notpresent': 0, 'present': 1},
    'htn': {'no': 0, 'yes': 1},
    'dm': {'no': 0, 'yes': 1},
    'cad': {'no': 0, 'yes': 1},
    'appet': {'good': 0, 'poor': 1},
    'pe': {'no': 0, 'yes': 1},
    'ane': {'no': 0, 'yes': 1},
    'classification': {'notckd': 0, 'ckd': 1}
}

for col, mapping in mapping_dict.items():
    if col in ckd_df.columns:
        ckd_df[col] = ckd_df[col].replace(mapping)

print("Categorical columns encoded")

Categorical columns encoded


In [54]:
# Fill missing numeric values with median
for col in ckd_df.select_dtypes(include=['float64', 'int64']).columns:
    ckd_df[col] = ckd_df[col].fillna(ckd_df[col].median())

print("Numeric missing values filled with median")

Numeric missing values filled with median


In [55]:
# Step 13: Final dataset check
print("Dataset shape after cleaning:")
print(ckd_df.shape)

print("\nData types after cleaning:")
print(ckd_df.dtypes)

print("\nMissing values after cleaning:")
print(ckd_df.isnull().sum())

Dataset shape after cleaning:
(400, 25)

Data types after cleaning:
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc                object
pc                 object
pcc                object
ba                 object
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv               float64
wc                float64
rc                float64
htn                object
dm                 object
cad                object
appet              object
pe                 object
ane                object
classification     object
dtype: object

Missing values after cleaning:
age                 0
bp                  0
sg                  0
al                  0
su                  0
rbc               152
pc                 65
pcc                 4
ba                  4
bgr                 0
bu              

In [56]:
ckd_df.info()
ckd_df.head()
ckd_df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             400 non-null    float64
 1   bp              400 non-null    float64
 2   sg              400 non-null    float64
 3   al              400 non-null    float64
 4   su              400 non-null    float64
 5   rbc             248 non-null    object 
 6   pc              335 non-null    object 
 7   pcc             396 non-null    object 
 8   ba              396 non-null    object 
 9   bgr             400 non-null    float64
 10  bu              400 non-null    float64
 11  sc              400 non-null    float64
 12  sod             400 non-null    float64
 13  pot             400 non-null    float64
 14  hemo            400 non-null    float64
 15  pcv             400 non-null    float64
 16  wc              400 non-null    float64
 17  rc              400 non-null    float64
 18  h

,age,bp,sg,al,su,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc
count,400.000000,400.000000,400.000000,400.00000,400.000000,400.000000,400.000000,400.000000,400.000000,400.000000,400.00000,400.000000,400.000000,400.000000
mean,51.562500,76.575000,1.017712,0.90000,0.395000,145.062500,56.693000,2.997125,137.631250,4.577250,12.54250,39.082500,8298.500000,4.737750
std,16.982996,13.489785,0.005434,1.31313,1.040038,75.260774,49.395258,5.628886,9.206332,2.821357,2.71649,8.162245,2529.593814,0.841439
min,2.000000,50.000000,1.005000,0.00000,0.000000,22.000000,1.500000,0.400000,4.500000,2.500000,3.10000,9.000000,2200.000000,2.100000
25%,42.000000,70.000000,1.015000,0.00000,0.000000,101.000000,27.000000,0.900000,135.000000,4.000000,10.87500,34.000000,6975.000000,4.500000
50%,55.000000,80.000000,1.020000,0.00000,0.000000,121.000000,42.000000,1.300000,138.000000,4.400000,12.65000,40.000000,8000.000000,4.800000
75%,64.000000,80.000000,1.020000,2.00000,0.000000,150.000000,61.750000,2.725000,141.000000,4.800000,14.62500,44.000000,9400.000000,5.100000
max,90.000000,180.000000,1.025000,5.00000,5.000000,490.000000,391.000000,76.000000,163.000000,47.000000,17.80000,54.000000,26400.000000,8.000000


In [57]:
# Fix categorical columns properly

# Convert object columns to clean lowercase text
for col in ckd_df.select_dtypes(include=['object']).columns:
    ckd_df[col] = ckd_df[col].astype(str).str.lower().str.strip()

# Replace bad text variations
ckd_df = ckd_df.replace({
    '\tyes': 'yes',
    '\tno': 'no',
    ' yes': 'yes',
    ' no': 'no',
    'ckd\t': 'ckd',
    'not ckd': 'notckd'
})

# Map categorical values
mapping_dict = {
    'rbc': {'normal': 0, 'abnormal': 1},
    'pc': {'normal': 0, 'abnormal': 1},
    'pcc': {'notpresent': 0, 'present': 1},
    'ba': {'notpresent': 0, 'present': 1},
    'htn': {'no': 0, 'yes': 1},
    'dm': {'no': 0, 'yes': 1},
    'cad': {'no': 0, 'yes': 1},
    'appet': {'good': 0, 'poor': 1},
    'pe': {'no': 0, 'yes': 1},
    'ane': {'no': 0, 'yes': 1},
    'classification': {'notckd': 0, 'ckd': 1}
}

for col, mapping in mapping_dict.items():
    if col in ckd_df.columns:
        ckd_df[col] = ckd_df[col].replace(mapping)

# Convert mapped columns to numeric
mapped_cols = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']

for col in mapped_cols:
    if col in ckd_df.columns:
        ckd_df[col] = pd.to_numeric(ckd_df[col], errors='coerce')

# Fill missing values
for col in ckd_df.columns:
    if ckd_df[col].dtype in ['float64', 'int64']:
        ckd_df[col] = ckd_df[col].fillna(ckd_df[col].median())
    else:
        ckd_df[col] = ckd_df[col].fillna(ckd_df[col].mode()[0])

print("Missing values after final fixing:")
print(ckd_df.isnull().sum())

print("\nData types after final fixing:")
print(ckd_df.dtypes)

Missing values after final fixing:
age               0
bp                0
sg                0
al                0
su                0
rbc               0
pc                0
pcc               0
ba                0
bgr               0
bu                0
sc                0
sod               0
pot               0
hemo              0
pcv               0
wc                0
rc                0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
classification    0
dtype: int64

Data types after final fixing:
age               float64
bp                float64
sg                float64
al                float64
su                float64
rbc               float64
pc                float64
pcc               float64
ba                float64
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hemo              float64
pcv               float64
wc  

In [58]:
ckd_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             400 non-null    float64
 1   bp              400 non-null    float64
 2   sg              400 non-null    float64
 3   al              400 non-null    float64
 4   su              400 non-null    float64
 5   rbc             400 non-null    float64
 6   pc              400 non-null    float64
 7   pcc             400 non-null    float64
 8   ba              400 non-null    float64
 9   bgr             400 non-null    float64
 10  bu              400 non-null    float64
 11  sc              400 non-null    float64
 12  sod             400 non-null    float64
 13  pot             400 non-null    float64
 14  hemo            400 non-null    float64
 15  pcv             400 non-null    float64
 16  wc              400 non-null    float64
 17  rc              400 non-null    float64
 18  h

In [59]:
# Save cleaned dataset for later modeling
ckd_df.to_csv("../data/chronic_kidney_disease/ckd_cleaned.csv", index=False)

print("Cleaned CKD dataset saved successfully")

Cleaned CKD dataset saved successfully
